# QML on MNIST Classification

## Overview

The aim of this tutorial is not about the machine learning perspective on better design of QML method for MNIST classification. Instead, we use a simple parameterized circuit and demonstrate the QML-related technical ingredients of ``tensorcircuit``. Nevertheless, this note is by no means a good practice on QML.

[WIP note]

## Setup

In [1]:
from functools import partial
import numpy as np
import tensorflow as tf
import jax
from jax import config

config.update("jax_enable_x64", True)
from jax import numpy as jnp
import optax
import tensorcircuit as tc

In [2]:
tc.set_backend("tensorflow")
tc.set_dtype("complex128")

('complex128', 'float64')

## Data Processing

We utilize MNIST data and resize them to 3*3 to fit into a 9-qubit circuit.
The testbed we use is a binary classification task, digit 1 vs. 5.
And since this tutorial is not about good practice on QML, we leave the validation set away.
And we only collect 100 data points for a small demo.

In [3]:
# numpy data
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train[..., np.newaxis] / 255.0


def filter_pair(x, y, a, b):
    keep = (y == a) | (y == b)
    x, y = x[keep], y[keep]
    y = y == a
    return x, y


x_train, y_train = filter_pair(x_train, y_train, 1, 5)
x_train_small = tf.image.resize(x_train, (3, 3)).numpy()
x_train_bin = np.array(x_train_small > 0.5, dtype=np.float32)
x_train_bin = np.squeeze(x_train_bin)[:100]

In [4]:
# tensorflow data

x_train_tf = tf.reshape(tf.constant(x_train_bin, dtype=tf.float64), [-1, 9])
y_train_tf = tf.constant(y_train[:100], dtype=tf.float64)

# jax data

x_train_jax = jnp.array(x_train_bin, dtype=np.float64).reshape([100, -1])
y_train_jax = jnp.array(y_train[:100], dtype=np.float64).reshape([100])

## Using ``vectorized_value_and_grad`` API

In [5]:
nlayers = 3


def qml_loss(x, y, weights, nlayers):
    n = 9
    weights = tc.backend.cast(weights, "complex128")
    x = tc.backend.cast(x, "complex128")
    c = tc.Circuit(n)
    for i in range(n):
        c.rx(i, theta=x[i])
    for j in range(nlayers):
        for i in range(n - 1):
            c.cnot(i, i + 1)
        for i in range(n):
            c.rx(i, theta=weights[2 * j, i])
            c.ry(i, theta=weights[2 * j + 1, i])
    ypred = c.expectation([tc.gates.z(), (4,)])
    ypred = tc.backend.real(ypred)
    ypred = (tc.backend.real(ypred) + 1) / 2.0
    return -y * tc.backend.log(ypred) - (1 - y) * tc.backend.log(1 - ypred), ypred

In [6]:
def get_qml_vvag():
    qml_vvag = tc.backend.vectorized_value_and_grad(
        qml_loss, argnums=(2,), vectorized_argnums=(0, 1), has_aux=True
    )
    qml_vvag = tc.backend.jit(qml_vvag, static_argnums=(3,))
    return qml_vvag


qml_vvag = get_qml_vvag()
qml_vvag(x_train_tf, y_train_tf, tf.ones([nlayers * 2, 9], dtype=tf.float64), nlayers)

((<tf.Tensor: shape=(100,), dtype=float64, numpy=
  array([0.8433698 , 0.56257199, 0.54653163, 0.56257199, 0.82036163,
         0.56257199, 0.56257199, 0.58030506, 0.82036163, 0.56257199,
         0.82036163, 0.56257199, 0.82036163, 0.56257199, 0.54653163,
         0.54653163, 0.56257199, 0.56257199, 0.58030506, 0.82036163,
         0.54653163, 0.56257199, 0.56257199, 0.56257199, 0.56257199,
         0.56257199, 0.56257199, 0.85182866, 0.56257199, 0.82036163,
         0.82036163, 0.56257199, 0.8433698 , 0.56257199, 0.8433698 ,
         0.56257199, 0.85182866, 0.56257199, 0.82036163, 0.54653163,
         0.56257199, 0.56257199, 0.56257199, 0.56257199, 0.8433698 ,
         0.58030506, 0.56257199, 0.82036163, 0.8433698 , 0.8433698 ,
         0.54653163, 0.56257199, 0.82036163, 0.86501404, 0.56257199,
         0.56257199, 0.8433698 , 0.56257199, 0.85182866, 0.82036163,
         0.82036163, 0.56257199, 0.82036163, 0.56257199, 0.56257199,
         0.56257199, 0.82036163, 0.8433698 , 0.843369

In [7]:
# %timeit qml_vvag(x_train_tf, y_train_tf, tf.ones([nlayers*2, 9], dtype=tf.float64), nlayers)

### Jax Backend Compatibility

In [8]:
tc.set_backend("jax")

jax_backend

In [9]:
qml_vvag = get_qml_vvag()
qml_vvag(
    x_train_jax, y_train_jax, jnp.ones([nlayers * 2, 9], dtype=np.float64), nlayers
)

((Array([0.8433698 , 0.56257199, 0.54653163, 0.56257199, 0.82036163,
         0.56257199, 0.56257199, 0.58030506, 0.82036163, 0.56257199,
         0.82036163, 0.56257199, 0.82036163, 0.56257199, 0.54653163,
         0.54653163, 0.56257199, 0.56257199, 0.58030506, 0.82036163,
         0.54653163, 0.56257199, 0.56257199, 0.56257199, 0.56257199,
         0.56257199, 0.56257199, 0.85182866, 0.56257199, 0.82036163,
         0.82036163, 0.56257199, 0.8433698 , 0.56257199, 0.8433698 ,
         0.56257199, 0.85182866, 0.56257199, 0.82036163, 0.54653163,
         0.56257199, 0.56257199, 0.56257199, 0.56257199, 0.8433698 ,
         0.58030506, 0.56257199, 0.82036163, 0.8433698 , 0.8433698 ,
         0.54653163, 0.56257199, 0.82036163, 0.86501404, 0.56257199,
         0.56257199, 0.8433698 , 0.56257199, 0.85182866, 0.82036163,
         0.82036163, 0.56257199, 0.82036163, 0.56257199, 0.56257199,
         0.56257199, 0.82036163, 0.8433698 , 0.8433698 , 0.82036163,
         0.56257199, 0.56257199, 0

In [10]:
# %timeit qml_vvag(x_train_jax, y_train_jax, jnp.ones([nlayers * 2, 9], dtype=np.float64), nlayers)

### Training Using ``tf.data``

In [11]:
# switch back to tensorflow
tc.set_backend("tensorflow")
qml_vvag = get_qml_vvag()
qml_vvag = tc.backend.jit(qml_vvag, static_argnums=(3,))

In [12]:
mnist_data = (
    tf.data.Dataset.from_tensor_slices((x_train_tf, y_train_tf))
    .repeat(200)
    .shuffle(100)
    .batch(32)
)

In [13]:
opt = tf.keras.optimizers.Adam(1e-2)
w = tf.Variable(
    initial_value=tf.random.normal(shape=(2 * nlayers, 9), stddev=0.5, dtype=tf.float64)
)
for i, (xs, ys) in zip(range(2000), mnist_data):
    (losses, ypreds), grad = qml_vvag(xs, ys, w, nlayers)
    if i % 20 == 0:
        print(tf.reduce_mean(losses))
    opt.apply_gradients([(grad[0], w)])

tf.Tensor(0.6882181339902962, shape=(), dtype=float64)
tf.Tensor(0.6814181009432645, shape=(), dtype=float64)


tf.Tensor(0.6139398088609226, shape=(), dtype=float64)
tf.Tensor(0.6484759331523565, shape=(), dtype=float64)


tf.Tensor(0.643387450151498, shape=(), dtype=float64)
tf.Tensor(0.6160787679905442, shape=(), dtype=float64)


tf.Tensor(0.7146481929542476, shape=(), dtype=float64)
tf.Tensor(0.6389381815122707, shape=(), dtype=float64)


tf.Tensor(0.6194927902308465, shape=(), dtype=float64)
tf.Tensor(0.6077618359706897, shape=(), dtype=float64)


...

tf.Tensor(0.5419974398095152, shape=(), dtype=float64)
tf.Tensor(0.5586035636860434, shape=(), dtype=float64)


tf.Tensor(0.6286397021937381, shape=(), dtype=float64)
tf.Tensor(0.6440015627994593, shape=(), dtype=float64)


tf.Tensor(0.6039320678146346, shape=(), dtype=float64)
tf.Tensor(0.5713502248153103, shape=(), dtype=float64)


tf.Tensor(0.565715083139002, shape=(), dtype=float64)
tf.Tensor(0.5949027348523617, shape=(), dtype=float64)


tf.Tensor(0.6012267756034322, shape=(), dtype=float64)
tf.Tensor(0.6109874317155778, shape=(), dtype=float64)


## Using ``tf.keras`` API

In [14]:
from tensorcircuit import keras


def qml_y(x, weights, nlayers):
    n = 9
    weights = tc.backend.cast(weights, "complex128")
    x = tc.backend.cast(x, "complex128")
    c = tc.Circuit(n)
    for i in range(n):
        c.rx(i, theta=x[i])
    for j in range(nlayers):
        for i in range(n - 1):
            c.cnot(i, i + 1)
        for i in range(n):
            c.rx(i, theta=weights[2 * j, i])
            c.ry(i, theta=weights[2 * j + 1, i])
    ypred = c.expectation([tc.gates.z(), (4,)])
    ypred = tc.backend.real(ypred)
    ypred = (tc.backend.real(ypred) + 1) / 2.0
    return ypred


ql = keras.QuantumLayer(partial(qml_y, nlayers=nlayers), [(2 * nlayers, 9)])

In [15]:
# keras interface with value and grad paradigm


@tf.function
def my_vvag(xs, ys):
    with tf.GradientTape() as tape:
        ypred = ql(xs)
        loss = tf.keras.losses.BinaryCrossentropy()(ys, ypred)
    return loss, tape.gradient(loss, ql.variables)


my_vvag(x_train_tf, y_train_tf)

(<tf.Tensor: shape=(), dtype=float32, numpy=0.6987088918685913>,
 [<tf.Tensor: shape=(6, 9), dtype=float64, numpy=
  array([[-1.10936817e-02, -1.33291538e-03, -5.87745373e-03,
          -6.35860588e-03, -2.93374022e-05, -5.94974417e-03,
          -2.07285971e-04, -1.51788304e-17,  3.46944695e-18],
         [ 1.12992936e-02,  2.18895149e-03,  3.53805373e-03,
           5.40332844e-03,  7.12623702e-03,  1.38455270e-03,
          -1.62189613e-02,  1.99493200e-17, -8.67361738e-19],
         [ 5.90694766e-03, -8.79280433e-03,  8.48293460e-04,
          -7.19331663e-03,  3.53495779e-06, -2.28551831e-04,
           8.89045781e-18,  2.16840434e-18, -2.81892565e-17],
         [-5.24045302e-03, -4.10061147e-03, -2.07908310e-03,
          -8.17067217e-03,  1.80390058e-02,  1.12861689e-04,
           4.77048956e-18,  3.03576608e-18,  1.73472348e-18],
         [-4.22838847e-18, -1.64798730e-17,  5.63785130e-18,
          -2.60208521e-18, -2.72541524e-03,  1.73472348e-18,
           6.81692116e-18, 

In [16]:
# %timeit my_vvag(x_train_tf, y_train_tf)

In [17]:
# keras interface with keras training paradigm

model = tf.keras.Sequential([ql])

model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(0.01),
    metrics=[tf.keras.metrics.BinaryAccuracy()],
)

model.fit(x_train_tf, y_train_tf, batch_size=32, epochs=100)

Epoch 1/100
4/4 19s 14ms/step - binary_accuracy: 0.6300 - loss: 0.6982
Epoch 2/100
4/4 0s 13ms/step - binary_accuracy: 0.6300 - loss: 0.6974
Epoch 3/100
4/4 0s 14ms/step - binary_accuracy: 0.6300 - loss: 0.6947
Epoch 4/100
4/4 0s 14ms/step - binary_accuracy: 0.6300 - loss: 0.6929
Epoch 5/100
4/4 0s 14ms/step - binary_accuracy: 0.6300 - loss: 0.6916
Epoch 6/100
4/4 0s 14ms/step - binary_accuracy: 0.6300 - loss: 0.6896
Epoch 7/100
4/4 0s 14ms/step - binary_accuracy: 0.6300 - loss: 0.6871
Epoch 8/100
4/4 0s 16ms/step - binary_accuracy: 0.6300 - loss: 0.6835
Epoch 9/100
4/4 0s 13ms/step - binary_accuracy: 0.6300 - loss: 0.6810
Epoch 10/100
4/4 0s 13ms/step - binary_accuracy: 0.6300 - loss: 0.6782
Epoch 11/100
4/4 0s 13ms/step - binary_accuracy: 0.6300 - loss: 0.6761
Epoch 12/100
4/4 0s 13ms/step - binary_accuracy: 0.6300 - loss: 0.6739
Epoch 13/100
4/4 0s 13ms/step - binary_accuracy: 0.6300 - loss: 0.6706
Epoch 14/100
4/4 0s 13ms/step - binary_accuracy: 0.6300 - loss: 0.6677
Epoch 15/100
4

### Quantum-Classical Hybrid Model in Keras

In [18]:
def qml_ys(x, weights, nlayers):
    n = 9
    weights = tc.backend.cast(weights, "complex128")
    x = tc.backend.cast(x, "complex128")
    c = tc.Circuit(n)
    for i in range(n):
        c.rx(i, theta=x[i])
    for j in range(nlayers):
        for i in range(n - 1):
            c.cnot(i, i + 1)
        for i in range(n):
            c.rx(i, theta=weights[2 * j, i])
            c.ry(i, theta=weights[2 * j + 1, i])
    ypreds = []
    for i in range(n):
        ypred = c.expectation([tc.gates.z(), (i,)])
        ypred = tc.backend.real(ypred)
        ypred = (tc.backend.real(ypred) + 1) / 2.0
        ypreds.append(ypred)
    return tc.backend.stack(ypreds)

In [19]:
ql = tc.keras.QuantumLayer(partial(qml_ys, nlayers=nlayers), [(2 * nlayers, 9)])
model = tf.keras.Sequential([ql, tf.keras.layers.Dense(1, activation="sigmoid")])

In [20]:
model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(0.01),
    metrics=[tf.keras.metrics.BinaryAccuracy()],
)

model.fit(x_train_tf, y_train_tf, batch_size=32, epochs=100)

Epoch 1/100
4/4 24s 21ms/step - binary_accuracy: 0.3700 - loss: 0.9667
Epoch 2/100
4/4 0s 21ms/step - binary_accuracy: 0.3700 - loss: 0.8628
Epoch 3/100
4/4 0s 21ms/step - binary_accuracy: 0.3700 - loss: 0.7793
Epoch 4/100
4/4 0s 19ms/step - binary_accuracy: 0.3700 - loss: 0.7279
Epoch 5/100
4/4 0s 21ms/step - binary_accuracy: 0.4400 - loss: 0.6928
Epoch 6/100
4/4 0s 19ms/step - binary_accuracy: 0.6300 - loss: 0.6705
Epoch 7/100
4/4 0s 19ms/step - binary_accuracy: 0.6300 - loss: 0.6634
Epoch 8/100
4/4 0s 20ms/step - binary_accuracy: 0.6300 - loss: 0.6582
Epoch 9/100
4/4 0s 19ms/step - binary_accuracy: 0.6300 - loss: 0.6574
Epoch 10/100
4/4 0s 19ms/step - binary_accuracy: 0.6300 - loss: 0.6602
Epoch 11/100
4/4 0s 19ms/step - binary_accuracy: 0.6300 - loss: 0.6603
Epoch 12/100
4/4 0s 19ms/step - binary_accuracy: 0.6300 - loss: 0.6582
Epoch 13/100
4/4 0s 22ms/step - binary_accuracy: 0.6300 - loss: 0.6569
Epoch 14/100
4/4 0s 21ms/step - binary_accuracy: 0.6300 - loss: 0.6558
Epoch 15/100
4

### Hybrid Model in Jax

In [21]:
tc.set_backend("jax")

jax_backend

In [22]:
key = jax.random.PRNGKey(42)
key, *subkeys = jax.random.split(key, num=4)
params = {
    "qweights": jax.random.normal(subkeys[0], shape=[nlayers * 2, 9]),
    "cweights:w": jax.random.normal(subkeys[1], shape=[9]),
    "cweights:b": jax.random.normal(subkeys[2], shape=[1]),
}

In [23]:
def qml_hybrid_loss(x, y, params, nlayers):
    weights = params["qweights"]
    w = params["cweights:w"]
    b = params["cweights:b"]
    ypred = qml_ys(x, weights, nlayers)
    ypred = tc.backend.reshape(ypred, [-1, 1])
    ypred = w @ ypred + b
    ypred = jax.nn.sigmoid(ypred)
    ypred = ypred[0]
    loss = -y * tc.backend.log(ypred) - (1 - y) * tc.backend.log(1 - ypred)
    return loss

In [24]:
qml_hybrid_loss_vag = tc.backend.jit(
    tc.backend.vvag(qml_hybrid_loss, vectorized_argnums=(0, 1), argnums=2),
    static_argnums=3,
)

In [25]:
qml_hybrid_loss_vag(x_train_jax, y_train_jax, params, nlayers)

(Array([0.42296214, 1.06451071, 1.06809695, 1.06451071, 0.42617817,
        0.99293054, 1.06451071, 1.05843045, 0.42617817, 0.99293054,
        0.42617817, 1.06451071, 0.42617817, 1.06451071, 1.06809695,
        1.00047892, 1.06451071, 1.06451071, 1.05843045, 0.42617817,
        1.00047892, 1.06451071, 1.06451071, 0.99293054, 1.06451071,
        1.06451071, 0.99293054, 0.41962206, 1.06451071, 0.47539413,
        0.42617817, 0.99293054, 0.42296214, 0.99293054, 0.42296214,
        1.06451071, 0.46151499, 1.06451071, 0.42617817, 1.06809695,
        1.06451071, 1.06451071, 0.99293054, 0.99293054, 0.42296214,
        0.97190916, 0.99293054, 0.47539413, 0.42296214, 0.46281253,
        1.06809695, 0.99293054, 0.42617817, 0.45839653, 1.06451071,
        1.06451071, 0.42296214, 1.06451071, 0.41962206, 0.47539413,
        0.42617817, 0.99293054, 0.42617817, 1.06451071, 0.99293054,
        1.06451071, 0.47539413, 0.46281253, 0.46281253, 0.42617817,
        1.06451071, 1.06451071, 1.06451071, 1.06

In [26]:
optimizer = optax.adam(5e-3)
opt_state = optimizer.init(params)
for i, (xs, ys) in zip(range(2000), mnist_data):  # using tf data loader here
    xs = xs.numpy()
    ys = ys.numpy()
    v, grads = qml_hybrid_loss_vag(xs, ys, params, nlayers)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    if i % 30 == 0:
        print(jnp.mean(v))

0.7595997774524819


0.6994468456477553


0.6051339455522041


0.6907751908593655


0.6504548087589277


...

0.4175060645911137


0.5555623797141207


0.49696723224154776


0.49861575232585675


0.6292010184536571
